# 03 · Encode the policy map as a QUBO

## Question

How can we represent the policy-map problem with binary variables?

## Concept

Use the illustrative scores from Notebook 01. In the order $q_{11},q_{12},q_{21},q_{22}$, a bit is 1 when its policy is selected for its regime. The constraints are $q_{11}+q_{12}=1$ and $q_{21}+q_{22}=1$.

$$H_\mathrm{perf}=-\sum_{ij}J_{ij}q_{ij},\qquad
H_\mathrm{onehot}=A\sum_i(\sum_j q_{ij}-1)^2.$$

The minus sign turns performance maximization into energy minimization. Because $q^2=q$, each constraint term expands to $A(1-q_{i1}-q_{i2}+2q_{i1}q_{i2})$. Adding both terms gives a quadratic unconstrained binary objective (QUBO). Here $A=5$ is enough: all performance contributions are nonnegative and every infeasible state pays at least 5, above the best feasible energy of 2.

## Check / exercise

Before running, predict the four feasible strings and calculate the energy of 0000. What changes if you set $A=0$?

## Worked example

In [1]:
from itertools import product

from IPython.display import Markdown, display

J = [[-1.0, -3.0], [-2.5, -1.0]]
A = 5.0
states = []
for bits in product((0, 1), repeat=4):
    q11, q12, q21, q22 = bits
    row_counts = (q11 + q12, q21 + q22)
    selected = [" + ".join(label for label, bit in zip(("π₁", "π₂"), row) if bit)
                or "none" for row in (bits[:2], bits[2:])]
    performance = -(J[0][0]*q11 + J[0][1]*q12 + J[1][0]*q21 + J[1][1]*q22)
    penalty = A * sum((count - 1)**2 for count in row_counts)
    states.append(("".join(map(str, bits)), *selected, row_counts == (1, 1),
                   performance, penalty, performance + penalty))

## Result

In [2]:
lines = ["| Bits | ρ₁ policies | ρ₂ policies | Feasible? | Performance term | Penalty | Energy |",
         "|---|---|---|---|---:|---:|---:|"]
for bits, first, second, feasible, performance, penalty, energy in states:
    flag = "**YES**" if feasible else "—"
    lines.append(f"| `{bits}` | {first} | {second} | {flag} | {performance:.1f} | {penalty:.1f} | {energy:.1f} |")
display(Markdown("\n".join(lines)))
feasible_count = sum(row[3] for row in states)
best = min(states, key=lambda row: row[-1])
assert feasible_count == 4 and best[0] == "1001" and best[-1] == 2
print(f"{feasible_count} of {len(states)} states are feasible. Exact ground state: {best[0]}, energy {best[-1]:g}.")

| Bits | ρ₁ policies | ρ₂ policies | Feasible? | Performance term | Penalty | Energy |
|---|---|---|---|---:|---:|---:|
| `0000` | none | none | — | 0.0 | 10.0 | 10.0 |
| `0001` | none | π₂ | — | 1.0 | 5.0 | 6.0 |
| `0010` | none | π₁ | — | 2.5 | 5.0 | 7.5 |
| `0011` | none | π₁ + π₂ | — | 3.5 | 10.0 | 13.5 |
| `0100` | π₂ | none | — | 3.0 | 5.0 | 8.0 |
| `0101` | π₂ | π₂ | **YES** | 4.0 | 0.0 | 4.0 |
| `0110` | π₂ | π₁ | **YES** | 5.5 | 0.0 | 5.5 |
| `0111` | π₂ | π₁ + π₂ | — | 6.5 | 5.0 | 11.5 |
| `1000` | π₁ | none | — | 1.0 | 5.0 | 6.0 |
| `1001` | π₁ | π₂ | **YES** | 2.0 | 0.0 | 2.0 |
| `1010` | π₁ | π₁ | **YES** | 3.5 | 0.0 | 3.5 |
| `1011` | π₁ | π₁ + π₂ | — | 4.5 | 5.0 | 9.5 |
| `1100` | π₁ + π₂ | none | — | 4.0 | 10.0 | 14.0 |
| `1101` | π₁ + π₂ | π₂ | — | 5.0 | 5.0 | 10.0 |
| `1110` | π₁ + π₂ | π₁ | — | 6.5 | 5.0 | 11.5 |
| `1111` | π₁ + π₂ | π₁ + π₂ | — | 7.5 | 10.0 | 17.5 |

4 of 16 states are feasible. Exact ground state: 1001, energy 2.


**Exercise check.** The all-zero string pays two penalties, for energy 10. With zero penalty its energy becomes 0, lower than that of any valid map.

## Takeaway

Four of the 16 strings encode valid maps. The one-hot penalty makes the lowest-energy string encode the same optimum as classical enumeration.

## Next step

[Notebook 04](04_qubo_to_ising.ipynb) writes the same energy in terms of spins.

### Presentation asset

Reuse the 16-state table with the four feasible rows emphasized and the winning row marked.